# 02 — Feature Engineering

Build the three core joined tables and derive all features:
- **customer_base**: one row per account (static + behavioral + fraud flags)
- **customer_monthly**: one row per account per month (lag, rolling, ratio features)
- **transaction_base_enriched**: transaction-level fraud flags

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline, load_table_as_df
from src import preprocessing as pp
from src import features as feat

pd.set_option('display.max_columns', 50)

In [2]:
con, counts = load_pipeline(verbose=True)

Loading CSV files into DuckDB…
  ✓ account_dim: 18,070 rows
  ✓ statement_fact: 658,228 rows
  ✓ transaction_fact: 493,336 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ wrld_stor_tran_fact: 1,053,854 rows
  ✓ syf_id: 18,070 rows
  ✓ rams_batch_cur: 96,799 rows
  ✓ fraud_claim_case: 77 rows
  ✓ fraud_claim_tran: 202 rows
  ✓ transaction_base (union): 1,547,190 rows

Verifying row counts…
  ✓ account_dim: 18,070 rows (OK)
  ✓ statement_fact: 658,228 rows (OK)
  ✓ transaction_fact: 493,336 rows (OK)
  ✓ wrld_stor_tran_fact: 1,053,854 rows (OK)
  ✓ transaction_base: 1,547,190 rows (OK)
  ✓ syf_id: 18,070 rows (OK)
  ✓ rams_batch_cur: 96,799 rows (OK)
  ✓ fraud_claim_case: 77 rows (OK)
  ✓ fraud_claim_tran: 202 rows (OK)


## 1. Build `customer_base` — One Row Per Account

In [3]:
customer_base = feat.build_customer_base(con)
print(f"\ncustomer_base shape: {customer_base.shape}")
print(f"Unique accounts: {customer_base['current_account_nbr'].nunique()}")


customer_base shape: (18070, 55)
Unique accounts: 18070


In [4]:
# Verify key derived features
print("\n=== Derived Feature Summary ===")
derived_cols = [
    'delinquent_cycle_count', 'max_delinquency_level', 'risk_flag_count',
    'avg_monthly_sales_6m', 'sales_trend_slope',
    'utilization_ratio', 'otb_ratio', 'has_fraud_case'
]
for col in derived_cols:
    if col in customer_base.columns:
        s = customer_base[col]
        print(f"  {col}: mean={s.mean():.3f}, median={s.median():.3f}, "
              f"min={s.min():.3f}, max={s.max():.3f}")


=== Derived Feature Summary ===
  delinquent_cycle_count: mean=0.197, median=0.000, min=0.000, max=9.000
  max_delinquency_level: mean=0.127, median=0.000, min=0.000, max=4.000
  risk_flag_count: mean=0.055, median=0.000, min=0.000, max=6.000
  avg_monthly_sales_6m: mean=511.130, median=18.441, min=0.000, max=29842.050
  sales_trend_slope: mean=-0.308, median=0.000, min=-4539.378, max=3985.267
  utilization_ratio: mean=0.195, median=0.038, min=-1.289, max=1.680
  otb_ratio: mean=0.784, median=0.972, min=-0.546, max=2.369
  has_fraud_case: mean=0.004, median=0.000, min=0.000, max=1.000


In [5]:
# Fraud case flag distribution
fraud_counts = customer_base['has_fraud_case'].value_counts()
print(f"\nFraud case distribution:")
print(fraud_counts)
print(f"Fraud rate: {fraud_counts.get(1, 0) / len(customer_base) * 100:.2f}%")


Fraud case distribution:
has_fraud_case
0    17993
1       77
Name: count, dtype: int64
Fraud rate: 0.43%


## 2. Build `customer_monthly` — Account × Month

In [6]:
customer_monthly = feat.build_customer_monthly(con)
print(f"\ncustomer_monthly shape: {customer_monthly.shape}")
print(f"Unique accounts: {customer_monthly['current_account_nbr'].nunique()}")
print(f"Date range: {customer_monthly['month'].min()} to {customer_monthly['month'].max()}")


customer_monthly shape: (663105, 15)
Unique accounts: 17869
Date range: 2015-03-01 00:00:00 to 2025-03-01 00:00:00


In [7]:
# Add lag features
customer_monthly = feat.add_lag_features(customer_monthly)
print("\nLag features added:")
lag_cols = ['spend_lag_1', 'spend_lag_2', 'spend_lag_3', 
            'spend_rolling_mean_3', 'spend_rolling_std_3']
for col in lag_cols:
    if col in customer_monthly.columns:
        non_null = customer_monthly[col].notna().sum()
        print(f"  {col}: {non_null:,} non-null values")


Lag features added:
  spend_lag_1: 645,236 non-null values
  spend_lag_2: 628,097 non-null values
  spend_lag_3: 611,663 non-null values
  spend_rolling_mean_3: 645,236 non-null values
  spend_rolling_std_3: 663,105 non-null values


In [8]:
# Add spend-to-limit ratio
customer_monthly = feat.add_spend_to_limit_ratio(customer_monthly, customer_base)
print(f"\nspend_to_limit_ratio: mean={customer_monthly['spend_to_limit_ratio'].mean():.4f}")


spend_to_limit_ratio: mean=0.0164


In [9]:
# Monthly spend distribution
monthly_agg = customer_monthly.groupby('month').agg(
    total_spend=('total_spend', 'sum'),
    avg_spend=('total_spend', 'mean'),
    active_accounts=('current_account_nbr', 'nunique')
).reset_index()

fig = px.line(monthly_agg, x='month', y='total_spend',
              title='Total Monthly Spend Across All Accounts',
              labels={'total_spend': 'Total Spend ($)', 'month': 'Month'})
fig.update_traces(line=dict(width=3, color='#e74c3c'))
fig.update_layout(template='plotly_white', height=400)
fig.write_image("outputs/figures/feature_monthly_spend.png", scale=2)
fig.show()

## 3. Build `transaction_base_enriched` — Fraud-Flagged Transactions

In [10]:
txn_enriched = feat.build_transaction_enriched(con)
print(f"\ntransaction_base_enriched shape: {txn_enriched.shape}")
print(f"Flagged fraud transactions: {txn_enriched['is_fraud_txn'].sum()}")
print(f"Fraud transaction rate: {txn_enriched['is_fraud_txn'].mean()*100:.4f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


transaction_base_enriched shape: (1547190, 23)
Flagged fraud transactions: 3
Fraud transaction rate: 0.0002%


## 4. Build Q4 Feature Matrix

In [11]:
train_df, test_df, feature_cols, target_col = feat.build_q4_feature_matrix(
    customer_monthly, customer_base, cutoff_month='2024-10-01'
)

print(f"\nQ4 Feature Matrix:")
print(f"  Train: {train_df.shape} (months before Oct 2024)")
print(f"  Test:  {test_df.shape} (Oct-Dec 2024)")
print(f"  Features: {len(feature_cols)}")
print(f"  Feature names: {feature_cols}")


Q4 Feature Matrix:
  Train: (47880, 38) (months before Oct 2024)
  Test:  (49967, 38) (Oct-Dec 2024)
  Features: 28
  Feature names: ['spend_lag_1', 'spend_lag_2', 'spend_lag_3', 'spend_rolling_mean_3', 'spend_rolling_std_3', 'txn_count', 'avg_txn_amt', 'max_txn_amt', 'total_cash_advance', 'total_returns', 'prev_balance', 'spend_to_limit_ratio', 'cu_bhv_scr', 'cu_crd_bureau_scr', 'cu_crd_line', 'ca_current_utilz', 'ca_avg_utilz_lst_3_mnths', 'ca_avg_utilz_lst_6_mnths', 'ca_mob', 'ca_nsf_count_lst_12_months', 'ca_mnths_since_active', 'avg_monthly_sales_6m', 'sales_trend_slope', 'delinquent_cycle_count', 'max_delinquency_level', 'utilization_ratio', 'otb_ratio', 'has_fraud_case']


In [12]:
# Feature value ranges
print("\nFeature Statistics (Training Set):")
print(train_df[feature_cols].describe().round(2).to_string())


Feature Statistics (Training Set):
       spend_lag_1  spend_lag_2  spend_lag_3  spend_rolling_mean_3  spend_rolling_std_3  txn_count  avg_txn_amt  max_txn_amt  total_cash_advance  total_returns  prev_balance  spend_to_limit_ratio  cu_bhv_scr  cu_crd_bureau_scr  cu_crd_line  ca_current_utilz  ca_avg_utilz_lst_3_mnths  ca_avg_utilz_lst_6_mnths   ca_mob  ca_nsf_count_lst_12_months  ca_mnths_since_active  avg_monthly_sales_6m  sales_trend_slope  delinquent_cycle_count  max_delinquency_level  utilization_ratio  otb_ratio  has_fraud_case
count     47880.00     47880.00     47880.00              47880.00             47880.00   47880.00     47880.00     47880.00            47880.00       47880.00      47880.00              47880.00     47880.0            47880.0      47880.0           47880.0                   47880.0                   47880.0  47880.0                     47880.0                47880.0              47880.00           47880.00                47880.00               47880.00   

## 5. Build Anomaly Detection Features

In [13]:
anomaly_features = feat.build_anomaly_features(con)
print(f"\nAnomaly features shape: {anomaly_features.shape}")
print(anomaly_features.describe().round(2))


Anomaly features shape: (14283, 8)
       total_spend  txn_count  avg_txn_amt  max_txn_amt  total_foreign_amt  \
count     14283.00   14283.00     14283.00     14283.00           14283.00   
mean      14566.07     108.32       320.07      1867.21            1777.10   
std       30208.01     219.88       894.51      2905.00            5978.28   
min       -1687.04       1.00       -80.34        -0.13           -6220.11   
25%         909.92       7.00        61.74       183.67               0.00   
50%        2919.84      19.00       110.46       600.00              71.00   
75%       10864.12      70.00       218.41      2403.68             928.89   
max      687456.42    2442.00     25014.00     45000.00           98266.00   

       total_markup_fees  first_purchase_count  
count            14283.0              14283.00  
mean                 0.0                  0.31  
std                  0.0                  0.50  
min                  0.0                  0.00  
25%             

## 6. Save Intermediate DataFrames

In [15]:
# Save for downstream notebooks
# Cast mixed-type object columns to string so Parquet/Arrow can serialize them
for col in customer_base.select_dtypes(include=["object"]).columns:
    customer_base[col] = customer_base[col].astype(str)

customer_base.to_parquet("outputs/predictions/customer_base.parquet", index=False)
customer_monthly.to_parquet("outputs/predictions/customer_monthly.parquet", index=False)
anomaly_features.to_parquet("outputs/predictions/anomaly_features.parquet", index=False)

print("Saved:")
print("  - customer_base.parquet")
print("  - customer_monthly.parquet")
print("  - anomaly_features.parquet")

Saved:
  - customer_base.parquet
  - customer_monthly.parquet
  - anomaly_features.parquet


## Summary

| Table | Shape | Key Features |
|-------|-------|-------------|
| `customer_base` | One row per account | Sales trend, utilization ratio, delinquency, fraud flags |
| `customer_monthly` | Account × month | Lag features, rolling stats, spend-to-limit ratio |
| `transaction_enriched` | Per transaction | `is_fraud_txn` flag from fraud_claim_tran join |
| Q4 feature matrix | Train/Test split | Temporal: Jan-Sep train, Oct-Dec test |